# Convergence Diagnostics

MCMC chains require diagnostics to know whether they have converged and how many *effective* samples we have. This notebook walks through the three standard tools shipped with this package:

- **Autocorrelation function** (per-dimension)
- **Effective sample size (ESS)** via Geyer's initial monotone sequence estimator
- **R-hat** (Gelman-Rubin potential scale reduction factor) using multiple chains

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import torch
from torch.distributions.multivariate_normal import MultivariateNormal
import matplotlib.pyplot as plt

from mcmc_samplers import (
    DelayedRejectionAdaptiveMetropolis,
    HamiltonianMonteCarlo,
    SampleVisualizer,
    diagnostics,
    run_chains,
)

In [ ]:
class Banana:
    def __init__(self):
        self.mvn = MultivariateNormal(torch.zeros(2), torch.tensor([[1.0, 0.9], [0.9, 1.0]]))
    def __call__(self, x):
        y = torch.stack([x[..., 0], x[..., 1] + (x[..., 0] + 1.0) ** 2], dim=-1)
        return self.mvn.log_prob(y)

target = Banana()
x0_starts = [torch.tensor([0.0, 0.0]), torch.tensor([2.0, 2.0]), torch.tensor([-2.0, -2.0]), torch.tensor([1.0, -3.0])]

## Multi-chain run

We launch four DRAM chains from over-dispersed starting points using `run_chains`. The returned tensor has shape `(num_chains, N, dim)` and feeds directly into the diagnostics.

In [ ]:
N = 4000
burn_in = 1000

def factory(chain_idx):
    g = torch.Generator().manual_seed(chain_idx)
    return DelayedRejectionAdaptiveMetropolis(
        target=target,
        x0=x0_starts[chain_idx],
        cov=torch.eye(2),
        n0=200,
        generator=g,
    )

chains, log_probs = run_chains(factory, num_chains=4, N=N, burn_in=burn_in, show_progress=False)
print('chains shape:', tuple(chains.shape))

In [ ]:
vis = SampleVisualizer(chains)
print(vis.summary(labels=['x_1', 'x_2']))

## R-hat per dimension

Values close to 1 indicate good between- vs within-chain agreement. Common rule of thumb: anything above 1.1 deserves investigation.

In [ ]:
rhat = diagnostics.gelman_rubin(chains)
print('R-hat:', rhat.tolist())

## Autocorrelation plots

The `SampleVisualizer` overlays all chains on a single set of axes.

In [ ]:
vis.chains(labels=['$x_1$', '$x_2$'], titles=['Trace'])
vis.autocorrelation(max_lag=200, labels=['$x_1$', '$x_2$'], titles=['ACF'])
plt.show()

## Comparison: HMC ESS vs DRAM ESS

Gradient-based samplers should produce more uncorrelated samples per iteration. Here is a quick head-to-head.

In [ ]:
hmc = HamiltonianMonteCarlo(
    target=target, x0=torch.zeros(2), step_size=0.2, num_steps=5,
    generator=torch.Generator().manual_seed(0),
)
hmc_samples, _ = hmc(N, burn_in=burn_in, show_progress=False)

dram_ess = diagnostics.effective_sample_size(chains.reshape(-1, 2))
hmc_ess = diagnostics.effective_sample_size(hmc_samples)
print(f'DRAM ESS (all chains pooled): {dram_ess.tolist()}')
print(f'HMC  ESS:                     {hmc_ess.tolist()}')